# Interpretazione di una rete neurale per compliance normativa in ambito bancario

Banca Virtuosa, istituto di riferimento nel settore finanziario, ha identificato l'esigenza di migliorare la trasparenza e la comprensibilità dei modelli di intelligenza artificiale utilizzati nei propri sistemi. Per raggiungere questo obiettivo, Banca Virtuosa ha lanciato un progetto mirato all'implementazione di tecniche di Explainable AI (XAI), in conformità con la normativa vigente sulla trasparenza bancaria.

Attualmente, Banca Virtuosa utilizza modelli di classificazione pre-addestrati per analizzare e classificare dati finanziari critici. Tuttavia, la mancanza di trasparenza nelle decisioni di questi modelli può compromettere la fiducia dei clienti e limitare la capacità della banca di migliorare i propri sistemi in modo mirato. Identificare e correggere gli errori di classificazione è cruciale per garantire accuratezza e affidabilità nei servizi offerti.

## Benefici della Soluzione

**Trasparenza nelle Decisioni del Modello:** Implementando tecniche di XAI come Grad-CAM, LIME, SHAP, Integrated Gradients e Occlusion Maps, Banca Virtuosa sarà in grado di generare mappe di salienza che mostrano visivamente quali elementi influenzano le decisioni del modello. Questo incremento di trasparenza migliorerà la fiducia dei clienti e degli stakeholder, dimostrando l'affidabilità e la spiegabilità delle operazioni del sistema di classificazione.

**Miglioramento Continuo delle Performance:** Analizzando le mappe di salienza, Banca Virtuosa potrà identificare con precisione le aree in cui il modello commette errori, sia nelle classificazioni corrette che in quelle errate. Questa analisi dettagliata permetterà di apportare miglioramenti mirati al modello, ottimizzando le sue performance e riducendo il rischio di interpretazioni errate dei dati.

**Conformità Normativa:** Il progetto garantirà che le decisioni dei modelli di intelligenza artificiale siano spiegabili, in linea con i requisiti normativi vigenti. La trasparenza delle decisioni AI è essenziale per la conformità normativa e la governance aziendale, particolarmente in settori regolamentati come quello finanziario.

**Promozione dell'Innovazione:** L'utilizzo di tecniche avanzate di XAI all'interno di Banca Virtuosa promuoverà l'innovazione nel campo dell'intelligenza artificiale. Questo rafforzerà la posizione della banca come pioniere nell'adozione di tecnologie avanzate, consentendo di offrire ai clienti soluzioni sempre più sofisticate e affidabili.

## Dettagli del Progetto

**Fase 1: Utilizzo di un Modello di Classificazione Pre-Addestrato**
- Modello: Utilizzare un modello pre-addestrato, come DenseNet, dalla libreria torchvision.
- Dataset: Applicare il modello a un dataset di immagini, ad esempio MNIST, per esplorare le sue decisioni di classificazione.

**Fase 2: Generazione di Mappe di Salienza**
- Tecniche di XAI: Implementare tecniche come Grad-CAM, LIME, SHAP, Integrated Gradients e Occlusion Maps per generare mappe di salienza del modello.

**Fase 3: Report Finale**
- Descrizione del Dataset: Dettagliare l'origine, la struttura e le caratteristiche del dataset utilizzato.
- Analisi delle Mappe di Salienza: Confrontare le mappe di salienza per classi corrette ed errate per identificare e comprendere gli errori del modello.
- Sistema Spiegabile (Opzionale): Descrivere un sistema completamente spiegabile che potrebbe eseguire la stessa classificazione, offrendo ulteriori insights sulle decisioni del modello.

## Obiettivi del Progetto

- **Comprensione del Modello:** Utilizzare tecniche di XAI per ottenere una comprensione approfondita del funzionamento interno del modello pre-addestrato.
- **Visualizzazione delle Decisioni:** Visualizzare in modo chiaro e interpretabile quali elementi influenzano le decisioni del modello attraverso le mappe di salienza.
- **Identificazione degli Errori:** Analizzare le mappe di salienza per identificare e comprendere gli errori del modello, distinguendo tra classificazioni corrette ed errate.
- **Creazione di Sistemi Spiegabili:** Se possibile, sviluppare o descrivere un sistema completamente spiegabile che possa effettuare la stessa classificazione, fornendo ulteriori insights sulle decisioni del modello.

# Implementazione

Il progetto implementa un'analisi completa per BancaVirtuosa, al fine di migliorare la trasparenza, ottimizzare le performance dei modelli e garantire la conformità normativa. Con questo progetto, la banca mira a rafforzare la fiducia dei clienti, migliorare l'efficienza operativa e promuovere l'innovazione nel campo dell'intelligenza artificiale.

## Gestione delle dipendenze

Cominciamo quindi con l'analisi delle dipendenze necessarie e il relativo import nel notebook:

In [ ]:
#Standard library
import os
import sys
import time

#Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#Numpy
import numpy as np

#PyTorch
import torch
import torchvision
from torchvision.models import densenet161, DenseNet161_Weights
from torchsummary import summary
import torch.nn as nn
import torch.nn.functional as F

#Scikit Learn
from sklearn.metrics import classification_report, confusion_matrix

#Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

Importiamo inoltre il nostro modulo custom DNNHelper (https://github.com/crypto-infinity/dnnhelper):

In [ ]:
# Importing our custom Helper 
!git clone https://github.com/crypto-infinity/dnnhelper
%cd dnnhelper

In [ ]:
from dnnhelper import Helper, Transforms, EarlyStopping, Experiment, CrossValidation, Trainer

Definiamo, al termine del nostro setup, alcune costanti utili e alcune impostazioni del framework Deep Learning che useremo, PyTorch, come il device per il training dei modelli e il random seed per la riproducibilità:

In [ ]:
#PyTorch DEVICE setup

DEVICE = Helper.set_device()
print(f"Using device: {DEVICE}")

In [ ]:
#Random seed for reproducibility

SEED = 56

Helper.set_seed(SEED)
print(f"Random seed set to: {SEED}")

E costruiamo una prima pipeline per il processing delle immagini:

In [ ]:
# Preprocessing default template

default_preprocessing_pipeline = A.Compose([
            A.Resize(256, 256),
            A.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
            ToTensorV2(),
        ])

In [ ]:
iterations = [0,56,234,311]

## Import del dataset

Cominciamo ora ad importare il dataset che utilizzeremo, CIFAR10 (https://www.cs.toronto.edu/~kriz/cifar.html):

In [ ]:
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

n_classes = len(classes)

In [ ]:
trainset = torchvision.datasets.CIFAR10(root='./dataset', train=True,
                                        download=True, 
                                        transform=Transforms(default_preprocessing_pipeline))
train_dl = torch.utils.data.DataLoader(trainset, batch_size=32,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./dataset', train=False,
                                       download=True, 
                                       transform=Transforms(default_preprocessing_pipeline))
test_dl = torch.utils.data.DataLoader(testset, batch_size=32,
                                         shuffle=False, num_workers=2)

E conduciamone una prima esplorazione visiva, visualizzando alcuni dei sample del trainset:

In [ ]:
#First data exploration

trainset_defaultaug = trainset
trainset_defaultaug.transform = Transforms(default_preprocessing_pipeline)

for i in iterations:
    Helper.plot_images(trainset_defaultaug, trainset.classes, i)

del trainset_defaultaug

Verifichiamo anche che le distribuzioni dei samples siano bilanciate:

In [ ]:
Helper.plot_class_distribution(trainset, type="training")
Helper.plot_class_distribution(testset, type="test")

## Reti

Definiamo ora la rete DenseNet161 (https://docs.pytorch.org/vision/main/models/generated/torchvision.models.densenet161.html), come richiesto dal progetto, per effettuare alcune prime classificazioni del nostro dataset, analizzandone le metriche. Consideriamo CIFAR10 come un sottoproblema di ImageNet, perciò utilizzeremo gli stessi pesi.

In [ ]:
# Import DenseNet with ImageNet weights
densenet = densenet161(weights=DenseNet161_Weights.IMAGENET1K_V1).to(DEVICE)

# Freeze all layers
for param in densenet.parameters():
    param.requires_grad = False

In [ ]:
# Improved classifier with regularization
# We add a dropout layer and batch normalization to the classifier part of the model

class RegolarizedClassifier(nn.Module):
    """
    Regularized (in the classifier part) convolutional classifier using ResNet50 as backbone.
    """

    def __init__(self, densenet_backbone, in_features, n_classes):
        super().__init__()

        #backbone
        self.features = nn.Sequential(*list(densenet_backbone.children())[:-1])

        #fine-tuning
        self.pooling = nn.AdaptiveAvgPool2d((1,1))

        #Regularization
        self.dropout = nn.Dropout(p=0.2) #low dropout rate, an higher one would be too aggressive
        self.norm = nn.BatchNorm1d(in_features)

        #linear
        self.fc1 = nn.Linear(in_features, 2048)
        self.fc2 = nn.Linear(2048, n_classes)

    def forward(self, x):

        #backbone
        x = self.features(x) #[N, 2048, 8, 8]
        x = self.pooling(x) #[N, 2048, 1, 1]
        x = torch.flatten(x, 1) #[N, 2048]

        #dense
        x = F.relu(self.norm(self.fc1(x))) #normalizes the activations inputs
        x = self.dropout(x)
        x = self.fc2(x) #[N, 10]

        return x

In [ ]:
regolarized = RegolarizedClassifier(densenet, 2048, n_classes).to(DEVICE)

In [ ]:
print(regolarized)

In [ ]:
experiments = []

In [ ]:
experiments.append(
    Experiment(
        name = "autolr_regolarized",
        checkpoints_folder = 'models',
        checkpoint_name = "classifier.pt",
        model = regolarized,
        n_classes=n_classes,
        lr = 1e-4,
        lr_scheduler=True,
        lr_step=5,
        metrics = ["accuracy", "precision", "recall"],
        use_early_stopping = True,
        loss_fn = nn.CrossEntropyLoss,
        optimizer = torch.optim.Adam,
        epochs = 50,
        patience= 7,
        color = "#EB4310",
        alpha = .3,
    )
)

In [ ]:
# Train the model
start = time.time()

for exp in experiments:
    Trainer.fit(exp, train_dl, test_dl, verbose=True)

print(f"Tempo impiegato: {time.time() - start:.2f} secondi")

In [ ]:
# Test set evaluation

for exp in experiments:
    # Evaluate the model
    Helper.evaluate_experiment(exp, testset)